# TennisMyLife — 1903 Layout on Colab A100
VPS remains the controller/source of truth. Colab processes layout claims on the A100 and uploads `.layout.json` + `.layout.txt` immediately. Existing VPS layout outputs are preserved and skipped. Paddle/PaddleX run in an isolated virtualenv so Colab's preinstalled PyTorch/NCCL stack cannot conflict with the layout worker.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline /content/tml-layout-venv
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!python -m venv /content/tml-layout-venv
!/content/tml-layout-venv/bin/pip -q install --upgrade pip setuptools wheel
!/content/tml-layout-venv/bin/pip -q install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!/content/tml-layout-venv/bin/pip -q install paddlex==3.7.2 paddleocr==3.7.0 'paramiko>=3.5,<4'
print('Isolated Paddle layout environment ready')


In [ ]:
import subprocess, os, textwrap
VENV_PY='/content/tml-layout-venv/bin/python'
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
probe=r'''
import os, paddle
print('Paddle:',paddle.__version__,'compiled_cuda=',paddle.device.is_compiled_with_cuda(),flush=True)
if not paddle.device.is_compiled_with_cuda(): raise RuntimeError('Paddle CUDA build not active')
paddle.set_device('gpu:0')
print('Paddle device:',paddle.device.get_device(),'CPU cores:',os.cpu_count(),flush=True)
from paddlex import create_model
print('Warming PP-DocLayout_plus-L on A100...',flush=True)
m=create_model('PP-DocLayout_plus-L',device='gpu:0')
print('Layout model cache ready',flush=True)
del m
try: paddle.device.cuda.empty_cache()
except Exception: pass
'''
subprocess.run([VENV_PY,'-u','-c',probe],check=True)


In [ ]:
from google.colab import files
import base64
uploaded=files.upload()
if not uploaded: raise RuntimeError('No SSH key uploaded')
key_name,key_bytes=next(iter(uploaded.items()))
if key_name.endswith('.pub'): raise RuntimeError('Upload tml_colab_ed25519, not .pub')
KEY_B64=base64.b64encode(key_bytes).decode(); print('SSH key loaded:',key_name)


In [ ]:
import subprocess
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'; VPS_USER='andre'; VPS_PORT=2222
VPS_CLAIM='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_active_claim.tsv'
VPS_STOP='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_layout_quality_complete_1903.flag'
subprocess.run(['git','-C','/content/Tennis-OCR-Pipeline','pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C','/content/Tennis-OCR-Pipeline','rev-parse','--short','HEAD'],text=True).strip(); print('Code commit:',commit,flush=True)
cmd=[VENV_PY,'-u','/content/Tennis-OCR-Pipeline/colab/layout_pool.py','--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--claim',VPS_CLAIM,'--stop-flag',VPS_STOP,'--workers','4','--downloaders','8','--poll','10']
print('Starting isolated A100 layout pool: workers=4 downloaders=8',flush=True)
rc=subprocess.run(cmd).returncode
print('Layout worker finished rc=',rc,flush=True)


Leave the final cell running. It watches the VPS claim, skips outputs already completed on the VPS, processes the remaining pages on CUDA, then automatically picks up the promoted-layout claim. It exits when the VPS writes the quality-layout completion flag. The worker uses `/content/tml-layout-venv`, isolated from Colab's PyTorch/NCCL packages.
